In [ ]:
import streamlit as st
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

# 1️⃣ 학습된 딥러닝 모델 불러오기
model = tf.keras.models.load_model('Model.h5')

# 2️⃣ 예측 결과에 사용할 암 종류 이름들
labels = [
    'Colon Adenocarcinoma',
    'Colon Benign Tissue',
    'Lung Adenocarcinoma',
    'Lung Benign Tissue',
    'Lung Squamous Cell Carcinoma'
]

# 3️⃣ 이미지 전처리 함수
def preprocess_image(uploaded_file):
    img = image.load_img(uploaded_file, target_size=(224, 224))  # 모델 입력 크기로 맞춤
    img_array = image.img_to_array(img)  # 이미지 → 숫자 배열로 변환
    img_array = np.expand_dims(img_array, axis=0)  # (1, 224, 224, 3) 형태로 변경
    img_array /= 255.0  # 0~255 → 0~1 정규화
    return img_array

# 4️⃣ 암 종류 예측 함수
def predict_cancer(img_array):
    predictions = model.predict(img_array)[0]  # 모델 예측
    return {labels[i]: float(predictions[i]) for i in range(len(labels))}

# 5️⃣ 결과를 파이차트로 보여주는 함수
def plot_pie_chart(predictions):
    fig, ax = plt.subplots()
    ax.pie(predictions.values(), labels=predictions.keys(),
           autopct='%1.1f%%', startangle=140)
    ax.axis('equal')
    st.pyplot(fig)

# 6️⃣ Streamlit 웹 화면 구성
st.title("Cancer Type Prediction")
uploaded_file = st.file_uploader("Upload an image for prediction", type=["jpg", "png", "jpeg"])

# 7️⃣ 이미지가 업로드되었을 때 실행되는 전체 흐름
if uploaded_file is not None:
    st.image(uploaded_file, caption='Uploaded Image', use_column_width=True)

    img_array = preprocess_image(uploaded_file)     # 전처리
    predictions = predict_cancer(img_array)         # 예측

    st.subheader("Prediction Probabilities:")
    for label, probability in predictions.items():
        st.write(f"{label}: {probability*100:.2f}%")

    plot_pie_chart(predictions)                     # 시각화
